In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.utils import (
    CONFIG, load_trades, load_sentiment, merge_datasets,
    engineer_features, build_trader_profiles, cluster_traders,
    plot_pnl_by_sentiment, plot_winrate_by_sentiment,
    plot_trade_volume_by_sentiment, plot_top_vs_bottom_traders,
    plot_sentiment_lag_correlation, plot_archetypes,
    generate_summary_stats
)

print("✅ All imports successful")
print(f"📁 Working directory: {os.getcwd()}")


✅ All imports successful
📁 Working directory: d:\NEWprojects\hyperliquid-trader-analysis\notebooks


In [3]:
print("Loading datasets...")

trades = load_trades(CONFIG["trades_path"])
sentiment = load_sentiment(CONFIG["sentiment_path"])

df = merge_datasets(trades, sentiment)

print("Merge complete.")
df.head()

Loading datasets...


FileNotFoundError: Could not load trades file at '../data/compressed_data_csv'. Error: [Errno 2] No such file or directory: '../data/compressed_data_csv'

In [ ]:
print("=== DATASET OVERVIEW ===")
print(f"Trades      : {len(df):,}")
print(f"Accounts    : {df['account'].nunique()}")
print(f"Coins       : {df['coin'].nunique()}")
print(f"Date Range  : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"\n=== SENTIMENT DISTRIBUTION ===")
print(df['sentiment'].value_counts())
print(f"\n=== NULLS IN KEY COLUMNS ===")
print(df[['closed_pnl','sentiment','trade_size_usd' if 'trade_size_usd' in df.columns else 'size_usd']].isnull().sum())


In [ ]:
df       = engineer_features(df)
profiles = build_trader_profiles(df)
print("\nTrader profile preview:")
profiles[['account','total_trades','total_pnl','win_rate','avg_trade_size','liquidation_count']].head(5)


In [ ]:
profiles, cluster_centers, label_map = cluster_traders(
    profiles,
    n_clusters   = CONFIG["n_clusters"],
    random_state = CONFIG["random_state"]
)
print("\nArchetype distribution:")
print(profiles['archetype'].value_counts())
print("\nCluster centers (raw feature space):")
print(cluster_centers.round(2))


In [ ]:
# Checking whether market mood actually translates to trader outcomes
fig_path = plot_pnl_by_sentiment(df, CONFIG["figures_path"])

print("\nMean PnL by Sentiment:")
pnl_sent = df.groupby('sentiment')['closed_pnl'].agg(['mean','median','count'])
pnl_sent.columns = ['Avg PnL','Median PnL','Trade Count']
print(pnl_sent.round(2))
from IPython.display import Image
Image(fig_path)


In [ ]:
fig_path = plot_winrate_by_sentiment(df, CONFIG["figures_path"])

print("Win Rate by Sentiment:")
wr = df.groupby('sentiment')['profit_flag'].mean().mul(100).round(2)
print(wr)
from IPython.display import Image
Image(fig_path)


In [ ]:
fig_path = plot_trade_volume_by_sentiment(df, CONFIG["figures_path"])
from IPython.display import Image
Image(fig_path)


In [ ]:
fig_path = plot_top_vs_bottom_traders(profiles, CONFIG["top_n_traders"], CONFIG["figures_path"])

print("Top 10 traders summary:")
top10 = profiles.nlargest(10,'total_pnl')[['account','total_pnl','win_rate','avg_trade_size','liquidation_count']]
top10['account'] = top10['account'].str[:10] + '...'
print(top10.to_string(index=False))
from IPython.display import Image
Image(fig_path)


In [ ]:
fig_path, lag_corr = plot_sentiment_lag_correlation(df, CONFIG["figures_path"])
print(f"\nPearson r (lag-1 sentiment → next-day PnL): {lag_corr:.4f}")
if abs(lag_corr) < 0.15:
    print("→ Weak correlation. Sentiment alone is insufficient to predict daily PnL.")
    print("  This suggests other factors (coin selection, position sizing) dominate.")
elif lag_corr > 0:
    print("→ Positive lag correlation: greedy markets precede higher aggregate profits.")
else:
    print("→ Negative lag correlation: fear markets precede higher aggregate profits (contrarian signal).")
from IPython.display import Image
Image(fig_path)


In [ ]:
fig_path = plot_archetypes(profiles, CONFIG["figures_path"])
from IPython.display import Image
Image(fig_path)


In [ ]:
stats = generate_summary_stats(df, profiles)

print("=" * 55)
print("  HYPERLIQUID TRADER BEHAVIOR — KEY FINDINGS")
print("=" * 55)
print(f"  Dataset       : {stats['total_trades']:,} trades | {stats['total_accounts']} traders")
print(f"  Period        : {stats['date_range']}")
print(f"  Overall Win % : {stats['overall_win_rate']:.1%}")
print(f"  Overall Avg PnL: ${stats['overall_avg_pnl']:,.2f}")
print()
print("  PnL by Sentiment:")
for k, v in stats['pnl_by_sentiment'].items():
    print(f"    {k:<18}: ${v:>10,.2f}")
print()
print(f"  Best sentiment to trade : {stats['best_sentiment']}")
print(f"  Worst sentiment to trade: {stats['worst_sentiment']}")
print()
print(f"  Top trader PnL   : ${stats['top_trader_pnl']:,.2f}")
print(f"  Top trader WR    : {stats['top_trader_win_rate']:.1%}")
print()
print(f"  Total liquidations : {stats['total_liquidations']:,}")
print(f"  Liquidations by sentiment: {stats['liquidations_by_sentiment']}")
print("=" * 55)
